In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd

from db_functions import(
   connect_to_db,
   get_basic_info,
   get_additional_tables,
   get_categories,
   get_supplierrs,
   add_new_manual_id,
   get_all_products,
   get_product_history,
   place_reorder,
   get_pending_reorders,
   mark_reorder_as_received
)

st.siderbar.title("Inventory Management Dashboard")

option = st.sidebar.radio(
    "Select Option:",
    ["Basic Information","Operational Tasks"]
)

st.title("Invertory and Supply Chain Dashboard")

db=connect_to_db()
cursor=db.cursor(dictionary=True)

if option == "Basic Information":

    st.header("Basic Metrics")

    basic_infor=get_basic_info(cursor)

    cols=st.columns(3)
    keys=list(basic_info.keys())

    for i in range(3):
    cols[i].metric(
      label=keys[i],
      value=basic_info[keys[i]]
    )

    st.divider()


    tables=get_additional_tables(cursor)

    for labels,data in tables.items():

        st.header(labels)

        df=pd.DataFrame(data)

        st.dataframe(df)

        st.divider()

elif option == "Operational Tasks":
    st.header("Operational Tasks")

    selected_task=st.selectbox(
    "Choose a Task",
    [
    "Add New Product",
    "Product History",
    "Place Reorder",
    "Receive Reorder"
    ]
    )

    #ADD NEW PRODUCT

    if selected_task=='Add New Product":

       st.header("Add New Product")

       categories=get_categories(cursor)
       suppliers=get_suppliers(cursor)

       with st.form("Add_Product_Form");

         product_name=st.text_input("Product Name")

         product_category=st.selectbox(
             "Category",
             categories
         )
         product_price=st.number_input(
             "Price",
             min_value=0.00
         )
         product_stock=st.number_input(
             "Reorder Level",
             min_value=0,
             step=1
         )

         supplier_ids=[
             s["supplier_id"]
             for s in suppliers
         ]

         supplier_names=[
             s["supplier_name"]
             for s in suppliers
         ]

         supplier_id=st.selectbox(
             "Supplier",
             options=supplier_ids,
             format_func=lambda x:
             supplier_names[supplier_ids.index(x)]

         )
          submitted=st.form_submit_button(
              "Add Product"
          )
          if submitted:

              if not product_name:
                  st.error(
                      "Please Enter the Product Name."
                  )
              else:
                  try:
                      add_new_manual_id(
                          cursor,
                          db,
                          product_name,
                          product_category,
                          product_price,
                          product_stock,
                          product_level,
                          supplier_id
                      )

                      st.success(
                      f"Product{product_name}"
                      f"added successfully"
                      )
                  except Exception as e:

                      st.error(
                          f"Error adding the Product:{e}"
                      )
                      